# AutoML-M03: PyCaret

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## 🎯 Objetivo

Ejecutar **PyCaret AutoML** con validación cruzada (5-fold CV), comparativa
automática de ~15 modelos y selección del mejor para **Caso D** y **Caso D_strict**.

## 📊 Qué hace PyCaret

Framework AutoML completo: preprocesamiento automático, comparativa de modelos
con cross-validation, tuning de hiperparámetros y exportación del mejor modelo.
A diferencia de LazyPredict (M02), PyCaret **sí optimiza** y da métricas validadas por CV.

## 📝 Nota
Datos auditados (F3-M08). Sin leakage, sin constantes, sin traidoras.

## ⚠️ Requisitos
- **Kernel**: `Python (PyCaret)` — entorno conda `env_pycaret`
- Creado con `fautoml_setup_entornos.ipynb` (celda 4)

## 📦 Genera
- `data/automl/results_pycaret.parquet` — métricas de todos los modelos
- `docs/html/fase_automl/m03_pycaret.html` — documentación visual

## 🔄 Flujo
M02 (LazyPredict) → **M03 (PyCaret)** → M04 (H2O)

## 📌 Siguiente
`fautoml_m04_h2o.ipynb`

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
# Qué hace: Verifica/instala dependencias faltantes en el entorno conda,
#   carga librerías, detecta rutas y verifica datasets limpios.
# Requisitos: Kernel env_pycaret (creado con fautoml_setup_entornos.ipynb).
# ============================================================================

import sys, warnings, time, subprocess
from pathlib import Path
warnings.filterwarnings('ignore')

# --- Auto-verificación de dependencias ---
DEPENDENCIAS_REQUERIDAS = {
    'pycaret': 'pycaret[full]',
    'seaborn': 'seaborn',
    'matplotlib': 'matplotlib',
    'sklearn': 'scikit-learn',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'pyarrow': 'pyarrow',
    'jinja2': 'jinja2',
}

faltantes = []
for modulo, paquete_pip in DEPENDENCIAS_REQUERIDAS.items():
    try:
        __import__(modulo)
    except ImportError:
        faltantes.append(paquete_pip)

if faltantes:
    print(f'⚠️ Instalando dependencias faltantes: {faltantes}')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet'] + faltantes
    )
    print(f'✅ Instaladas: {faltantes}')
else:
    print('✅ Todas las dependencias disponibles')

# --- Detección de entorno (Colab / Local) ---
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent


sys.path.insert(0, str(ROOT))

# --- Imports principales ---
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, matthews_corrcoef, cohen_kappa_score, log_loss)

# --- Verificar PyCaret ---
import pycaret
from pycaret.classification import setup, compare_models, pull, predict_model, save_model
print(f'✅ PyCaret {pycaret.__version__} importado correctamente')

# --- Imports del proyecto ---
from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html import generar_kpis_html, generar_seccion_html, generar_html_navegacion_completa, guardar_html
from src.html.render import render_base_html

# --- Rutas y constantes ---
RUTA_FASE_AUTOML_HTML = RUTA_HTML / 'fase_automl'
crear_directorios([RUTA_AUTOML, RUTA_FASE_AUTOML_HTML])
fmt = formato_numero_es
RANDOM_STATE = 42
FRAMEWORK = 'PyCaret'

# DATASETS: Caso D y D_strict (limpios, sin leakage)
DATASETS = {
    'D': RUTA_AUTOML / 'df_exp_automl_D.parquet',
    'D_strict': RUTA_AUTOML / 'dataset_final_tfm.parquet',
}

info_entorno()

# --- Verificación anti-leakage ---
for caso, ruta in DATASETS.items():
    df_tmp = pd.read_parquet(ruta)
    n_cols = df_tmp.shape[1]
    del df_tmp
    print(f'  ✅ Caso {caso}: {ruta.name} ({n_cols} cols) — LIMPIO')
print('✅ Verificación anti-leakage superada')

✅ Todas las dependencias disponibles
✅ PyCaret 3.3.2 importado correctamente
✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Un

In [2]:
# ============================================================================
# CELDA 2: EJECUTAR PYCARET (CASO D + D_strict)
# ============================================================================
# Qué hace: Para cada caso, configura PyCaret (setup), compara ~15 modelos
#   con 5-fold CV, y recoge métricas en el esquema unificado.
# Nota: PyCaret hace su propio split interno (train_size=0.7).
#   Usamos session_id=42 para reproducibilidad.
# Genera: DataFrame unificado con todas las métricas por caso.
# ============================================================================

TARGET = 'abandono'
all_results = []

for caso, ruta in DATASETS.items():
    print(f'\n{"="*70}')
    print(f'CASO {caso}: {ruta.name}')
    print(f'{"="*70}')

    # --- Cargar datos ---
    df = pd.read_parquet(ruta)
    X = df.drop(columns=[TARGET])
    n_rows = len(df)
    n_cols = len(df.columns)
    n_feat = len(X.columns)
    print(f"Dataset: {n_rows:,} filas × {n_cols} columnas ({n_feat} features)")
    print(f'Target: {(df[TARGET]==1).sum():,} abandono ({(df[TARGET]==1).mean()*100:.1f}%)')

    # --- Setup PyCaret ---
    # PyCaret gestiona split, imputación, normalización internamente
    print(f'\n🔧 Configurando PyCaret...')
    t0_setup = time.time()
    s = setup(
        data=df,
        target=TARGET,
        session_id=RANDOM_STATE,
        train_size=0.7,
        fold=5,
        normalize=True,
        remove_multicollinearity=True,
        multicollinearity_threshold=0.95,
        verbose=False,
        html=False,
        log_experiment=False
    )
    print(f'  ✅ Setup completado en {time.time()-t0_setup:.1f}s')

    # --- Comparar modelos (5-fold CV) ---
    print(f'\n🤖 Comparando modelos (~15 algoritmos con 5-fold CV)...')
    print(f'   ⏱️ Esto puede tardar 5-15 minutos...')
    t0_comp = time.time()
    mejores = compare_models(
        n_select=5,
        sort='AUC',
        fold=5,
        verbose=True
    )
    df_comparativa = pull()
    t_comparar = time.time() - t0_comp
    print(f'  ✅ {len(df_comparativa)} modelos comparados en {t_comparar:.1f}s')

    # --- Convertir resultados al esquema unificado ---
    # PyCaret usa nombres de columnas diferentes: Accuracy, AUC, Recall, Prec., F1, Kappa, MCC
    print(f'\n🔄 Formateando resultados...')
    for idx, row in df_comparativa.iterrows():
        # PyCaret puede tener el nombre del modelo como índice o como columna 'Model'
        if 'Model' in df_comparativa.columns:
            model_name = row['Model']
        else:
            model_name = idx

        all_results.append({
            'caso': caso,
            'framework': FRAMEWORK,
            'model_name': model_name,
            'accuracy': row.get('Accuracy', 0),
            'balanced_accuracy': 0,  # PyCaret no reporta balanced_accuracy directamente
            'precision': row.get('Prec.', row.get('Precision', 0)),
            'recall': row.get('Recall', 0),
            'f1': row.get('F1', 0),
            'auc_roc': row.get('AUC', 0),
            'mcc': row.get('MCC', 0),
            'kappa': row.get('Kappa', 0),
            'log_loss': 999,  # PyCaret no lo reporta en compare_models
            'train_time_sec': round(row.get('TT (Sec)', 0), 2),
        })

    print(f'  ✅ {len(df_comparativa)} modelos procesados para caso {caso}')

    # --- Guardar mejor modelo del último caso (D_strict = producción) ---
    if caso == 'D_strict':
        mejor_modelo = mejores[0] if isinstance(mejores, list) else mejores
        ruta_modelo = RUTA_AUTOML / 'mejor_modelo_pycaret'
        save_model(mejor_modelo, str(ruta_modelo))
        print(f'  💾 Mejor modelo guardado: {ruta_modelo.name}.pkl')

# --- DataFrame unificado ---
df_resultados = pd.DataFrame(all_results)
df_resultados = df_resultados.sort_values(['caso', 'f1'], ascending=[True, False]).reset_index(drop=True)

# --- Ranking final por caso ---
for caso in DATASETS.keys():
    print(f'\n--- RANKING CASO {caso} (por F1, top 10) ---')
    mask = df_resultados['caso'] == caso
    print(df_resultados.loc[mask, ['model_name', 'accuracy', 'f1', 'auc_roc', 'mcc', 'train_time_sec']].head(10).to_string(index=False))

print(f'\n📊 Total: {len(df_resultados)} resultados ({len(df_resultados)//2} modelos × 2 casos)')


CASO D: df_exp_automl_D.parquet
Dataset: 33,621 filas × 22 columnas (21 features)
Target: 9,833 abandono (29.2%)

🔧 Configurando PyCaret...
  ✅ Setup completado en 14.5s

🤖 Comparando modelos (~15 algoritmos con 5-fold CV)...
   ⏱️ Esto puede tardar 5-15 minutos...


                                    Model  Accuracy     AUC  Recall   Prec.  \
catboost              CatBoost Classifier    0.8911  0.9441  0.7600  0.8517   
lightgbm  Light Gradient Boosting Machine    0.8900  0.9436  0.7606  0.8476   
rf               Random Forest Classifier    0.8839  0.9376  0.7250  0.8561   
gbc          Gradient Boosting Classifier    0.8732  0.9302  0.7177  0.8258   
et                 Extra Trees Classifier    0.8764  0.9297  0.6872  0.8622   
ada                  Ada Boost Classifier    0.8572  0.9102  0.7148  0.7788   
lr                    Logistic Regression    0.8469  0.8919  0.6798  0.7699   
ridge                    Ridge Classifier    0.8455  0.8893  0.6746  0.7688   
lda          Linear Discriminant Analysis    0.8443  0.8893  0.6904  0.7561   
svm                   SVM - Linear Kernel    0.8399  0.8833  0.6779  0.7523   
qda       Quadratic Discriminant Analysis    0.8281  0.8761  0.7446  0.6913   
knn                K Neighbors Classifier    0.8395 

                                    Model  Accuracy     AUC  Recall   Prec.  \
lightgbm  Light Gradient Boosting Machine    0.8845  0.9316  0.7331  0.8516   
catboost              CatBoost Classifier    0.8840  0.9305  0.7318  0.8506   
rf               Random Forest Classifier    0.8792  0.9264  0.7077  0.8544   
et                 Extra Trees Classifier    0.8674  0.9204  0.6653  0.8486   
gbc          Gradient Boosting Classifier    0.8701  0.9198  0.6933  0.8345   
ada                  Ada Boost Classifier    0.8550  0.8962  0.7009  0.7809   
lr                    Logistic Regression    0.8455  0.8843  0.6679  0.7731   
ridge                    Ridge Classifier    0.8436  0.8821  0.6600  0.7721   
lda          Linear Discriminant Analysis    0.8418  0.8821  0.6773  0.7562   
svm                   SVM - Linear Kernel    0.8366  0.8749  0.6720  0.7469   
qda       Quadratic Discriminant Analysis    0.8251  0.8687  0.7147  0.6955   
knn                K Neighbors Classifier    0.8366 

In [3]:
# ============================================================================
# CELDA 3: GUARDAR RESULTADOS
# ============================================================================
# Qué hace: Guarda métricas en parquet para la comparativa final (M06).
# Genera: data/automl/results_pycaret.parquet
# ============================================================================

ruta_resultados = RUTA_AUTOML / 'results_pycaret.parquet'
df_resultados.to_parquet(ruta_resultados, index=False)
print(f'💾 {ruta_resultados.name}: {len(df_resultados)} filas (caso D + D_strict)')

💾 results_pycaret.parquet: 30 filas (caso D + D_strict)


In [4]:
# ============================================================================
# CELDA 4: GRÁFICOS Y HTML
# ============================================================================
# Qué hace: Genera gráficos comparativos y documentación HTML.
# Genera: docs/html/fase_automl/m03_pycaret.html
# ============================================================================

graficos_b64 = {}
nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase_automl', modulo_activo='m03'
)

# --- Gráfico 1: Top 10 modelos por F1 (barras horizontales) ---
for caso in DATASETS.keys():
    mask = (df_resultados['caso'] == caso) & (df_resultados['model_name'] != 'Dummy Classifier')
    df_plot = df_resultados[mask].head(10).copy()
    df_plot = df_plot.sort_values('f1', ascending=True)

    fig, ax = plt.subplots(figsize=(12, 7))
    y_pos = np.arange(len(df_plot))

    bars = ax.barh(y_pos, df_plot['f1'], color='#38a169', alpha=0.85, height=0.6)
    ax.scatter(df_plot['auc_roc'], y_pos, color='#e53e3e', s=50, zorder=5, label='AUC-ROC')

    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_plot['model_name'], fontsize=9)
    ax.set_xlabel('Score')
    ax.set_title(f'PyCaret Caso {caso}: Top 10 Modelos (5-fold CV)', fontweight='bold', fontsize=14)
    ax.axvline(x=0.5, color='gray', ls='--', alpha=0.3)
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1.05)

    for bar, val in zip(bars, df_plot['f1']):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8, color='#2d3748')

    plt.tight_layout()
    graficos_b64[f'top10_{caso}'] = figura_a_base64(fig)
    plt.close()

# --- Gráfico 2: Top 5 métricas detalladas (barras agrupadas) ---
for caso in DATASETS.keys():
    mask = (df_resultados['caso'] == caso) & (df_resultados['model_name'] != 'Dummy Classifier')
    df_plot = df_resultados[mask].head(5).copy()

    metricas_plot = ['accuracy', 'f1', 'auc_roc', 'mcc', 'kappa']
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(df_plot))
    width = 0.15
    colores = ['#3182ce', '#38a169', '#e53e3e', '#805ad5', '#ed8936']

    for j, (met, col) in enumerate(zip(metricas_plot, colores)):
        ax.bar(x + j*width, df_plot[met], width, label=met, color=col)

    ax.set_xticks(x + width*2)
    ax.set_xticklabels(df_plot['model_name'], rotation=25, ha='right', fontsize=9)
    ax.set_ylabel('Score')
    ax.set_title(f'PyCaret Caso {caso}: Top 5 — Métricas Detalladas', fontweight='bold', fontsize=14)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.axhline(y=0.5, color='gray', ls='--', alpha=0.3)
    plt.tight_layout()
    graficos_b64[f'barras_{caso}'] = figura_a_base64(fig)
    plt.close()

# --- Construir HTML ---
contenido_html = ''

# Sección introductoria
contenido_html += generar_seccion_html('Metodología', f'''
<p><strong>PyCaret</strong> es un framework AutoML completo que automatiza preprocesamiento,
comparativa de modelos con cross-validation, y selección del mejor modelo.</p>
<div style="background:#f0fff4;padding:12px;border-radius:8px;margin-top:10px;border-left:4px solid #38a169;">
  <strong>✅ Validación:</strong> 5-fold cross-validation estratificada. Las métricas reflejan
  rendimiento generalizado, no sobreajuste a un split particular.
</div>
<div style="background:#EBF8FF;padding:12px;border-radius:8px;margin-top:10px;border-left:4px solid #3182ce;">
  <strong>📌 Config:</strong> normalize=True, remove_multicollinearity (umbral 0.95),
  session_id=42, train_size=0.7.
</div>
''', '🔬')

# Resultados por caso
for caso in DATASETS.keys():
    mask = df_resultados['caso'] == caso
    df_c = df_resultados[mask]
    mejor = df_c.iloc[0]
    n_modelos = len(df_c)

    # Tabla completa
    tabla = '<table style="width:100%;border-collapse:collapse;">\n'
    tabla += '<tr style="background:#38a169;color:white;">'
    for col in ['#', 'Modelo', 'Acc', 'Prec', 'Recall', 'F1', 'AUC', 'MCC', 'Kappa', 'Tiempo']:
        tabla += f'<th style="padding:8px;text-align:center;font-size:11px;">{col}</th>'
    tabla += '</tr>\n'

    for rank, (i, row) in enumerate(df_c.iterrows(), 1):
        bg = '#f7fafc' if rank % 2 == 0 else 'white'
        if rank <= 3:
            medallas = {1: '🥇', 2: '🥈', 3: '🥉'}
            rank_str = medallas[rank]
        else:
            rank_str = str(rank)

        tabla += f'<tr style="background:{bg};">'
        tabla += f'<td style="padding:6px;text-align:center;font-size:11px;">{rank_str}</td>'
        tabla += f'<td style="padding:6px;font-size:11px;">{row["model_name"]}</td>'
        for c in ['accuracy', 'precision', 'recall', 'f1', 'auc_roc', 'mcc', 'kappa']:
            v = row[c]
            color = '#38a169' if v > 0.7 else '#ed8936' if v > 0.5 else '#e53e3e'
            tabla += f'<td style="text-align:center;color:{color};font-size:11px;">{v:.4f}</td>'
        tabla += f'<td style="text-align:center;font-size:11px;">{row["train_time_sec"]:.1f}s</td></tr>\n'
    tabla += '</table>'

    # Gráficos
    graf_top10 = f'<img src="data:image/png;base64,{graficos_b64[f"top10_{caso}"]}" style="max-width:100%;border-radius:8px;">'
    graf_barras = f'<img src="data:image/png;base64,{graficos_b64[f"barras_{caso}"]}" style="max-width:100%;border-radius:8px;">'

    desc_caso = 'Alerta temprana (21 features)' if caso == 'D' else 'Producción (19 features)'
    contenido_html += generar_seccion_html(
        f'🤖 Caso {caso}: {desc_caso}',
        f'<p><strong>🏆 Mejor:</strong> {mejor["model_name"]} '
        f'(F1={mejor["f1"]:.4f}, AUC={mejor["auc_roc"]:.4f}, MCC={mejor["mcc"]:.4f})</p>'
        f'<p><strong>📊 Modelos evaluados:</strong> {n_modelos} (5-fold CV)</p>\n'
        f'{graf_top10}\n<br>\n{tabla}\n<br>\n{graf_barras}'
    )

# --- Sección comparativa vs anteriores ---
ruta_baselines = RUTA_AUTOML / 'results_baselines.parquet'
ruta_lazy = RUTA_AUTOML / 'results_lazypredict.parquet'
frameworks_prev = []
if ruta_baselines.exists():
    frameworks_prev.append(('Baselines (M01)', pd.read_parquet(ruta_baselines)))
if ruta_lazy.exists():
    frameworks_prev.append(('LazyPredict (M02)', pd.read_parquet(ruta_lazy)))

if frameworks_prev:
    comparativa = '<table style="width:100%;border-collapse:collapse;">\n'
    comparativa += '<tr style="background:#38a169;color:white;">'
    for col in ['Caso', 'Framework', 'Mejor Modelo', 'F1', 'AUC', 'MCC']:
        comparativa += f'<th style="padding:8px;text-align:center;">{col}</th>'
    comparativa += '</tr>\n'

    for caso in DATASETS.keys():
        # Frameworks anteriores
        for fw_name, df_fw in frameworks_prev:
            mask_fw = df_fw['caso'] == caso
            if 'model_name' in df_fw.columns:
                mask_fw = mask_fw & (~df_fw['model_name'].str.startswith('Dummy'))
            df_fw_caso = df_fw[mask_fw].sort_values('f1', ascending=False)
            if len(df_fw_caso) > 0:
                mejor_fw = df_fw_caso.iloc[0]
                comparativa += f'<tr style="background:#f7fafc;">'
                comparativa += f'<td style="padding:6px;text-align:center;">{caso}</td>'
                comparativa += f'<td style="padding:6px;text-align:center;">{fw_name}</td>'
                comparativa += f'<td style="padding:6px;">{mejor_fw["model_name"]}</td>'
                comparativa += f'<td style="text-align:center;">{mejor_fw["f1"]:.4f}</td>'
                comparativa += f'<td style="text-align:center;">{mejor_fw["auc_roc"]:.4f}</td>'
                comparativa += f'<td style="text-align:center;">{mejor_fw["mcc"]:.4f}</td></tr>\n'

        # PyCaret
        mask_pc = df_resultados['caso'] == caso
        mejor_pc = df_resultados[mask_pc].iloc[0]
        comparativa += f'<tr style="background:white;">'
        comparativa += f'<td style="padding:6px;text-align:center;">{caso}</td>'
        comparativa += f'<td style="padding:6px;text-align:center;"><strong>PyCaret (M03)</strong></td>'
        comparativa += f'<td style="padding:6px;"><strong>{mejor_pc["model_name"]}</strong></td>'
        comparativa += f'<td style="text-align:center;"><strong>{mejor_pc["f1"]:.4f}</strong></td>'
        comparativa += f'<td style="text-align:center;"><strong>{mejor_pc["auc_roc"]:.4f}</strong></td>'
        comparativa += f'<td style="text-align:center;"><strong>{mejor_pc["mcc"]:.4f}</strong></td></tr>\n'

    comparativa += '</table>'
    contenido_html += generar_seccion_html('🔄 Comparativa: M01 vs M02 vs M03', comparativa)

# --- KPIs ---
casos_k = list(DATASETS.keys())
mejor_d = df_resultados[df_resultados['caso']==casos_k[0]].iloc[0]
mejor_ds = df_resultados[df_resultados['caso']==casos_k[1]].iloc[0]
n_modelos_caso = len(df_resultados[df_resultados['caso']==casos_k[0]])

KPIS = [
    {'valor': str(n_modelos_caso), 'titulo': 'Modelos'},
    {'valor': f'{mejor_d["f1"]:.4f}', 'titulo': f'Mejor F1 (D)'},
    {'valor': f'{mejor_ds["f1"]:.4f}', 'titulo': f'Mejor F1 (D_strict)'},
    {'valor': '5-fold CV', 'titulo': 'Validación'},
]

# --- Renderizar HTML ---
html = render_base_html(
    titulo='M03: PyCaret', icono='🤖',
    subtitulo=f'AutoML con 5-fold CV (Caso D + D_strict)',
    nav_fases=nav_fases, nav_modulos=nav_modulos,
    contenido=generar_kpis_html(KPIS) + contenido_html,
    notebook_nombre='fautoml_m03_pycaret.ipynb', notebook_carpeta='fase_automl'
)
ruta_html = RUTA_FASE_AUTOML_HTML / 'm03_pycaret.html'
guardar_html(html, ruta_html)
print(f'\n✅ HTML: {ruta_html}')

✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m03_pycaret.html

✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m03_pycaret.html


In [5]:
# ============================================================================
# CELDA 5: RESUMEN
# ============================================================================

casos_k = list(DATASETS.keys())
mejor_d = df_resultados[df_resultados['caso']==casos_k[0]].iloc[0]
mejor_ds = df_resultados[df_resultados['caso']==casos_k[1]].iloc[0]
n_modelos_caso = len(df_resultados[df_resultados['caso']==casos_k[0]])

print('\n' + '='*60)
print('✅ AUTOML-M03 COMPLETADO')
print('='*60)
print(f'Framework: PyCaret {pycaret.__version__}')
print(f'Validación: 5-fold CV')
print(f'Modelos por caso: {n_modelos_caso}')
print(f'Caso D mejor: {mejor_d["model_name"]} (F1={mejor_d["f1"]:.4f}, AUC={mejor_d["auc_roc"]:.4f})')
print(f'Caso D_strict mejor: {mejor_ds["model_name"]} (F1={mejor_ds["f1"]:.4f}, AUC={mejor_ds["auc_roc"]:.4f})')
print(f'Resultados: {ruta_resultados}')
print(f'HTML: {ruta_html}')
print(f'\n🎯 Siguiente: fautoml_m04_h2o.ipynb')
print('='*60)


✅ AUTOML-M03 COMPLETADO
Framework: PyCaret 3.3.2
Validación: 5-fold CV
Modelos por caso: 15
Caso D mejor: CatBoost Classifier (F1=0.8032, AUC=0.9441)
Caso D_strict mejor: Light Gradient Boosting Machine (F1=0.7878, AUC=0.9316)
Resultados: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\results_pycaret.parquet
HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m03_pycaret.html

🎯 Siguiente: fautoml_m04_h2o.ipynb
